# Week 3 - Part 02: Data Profiling Script (CSV -> JSON/Markdown)

**Estimated time:** 90-120 minutes

This notebook builds a small data profiling script. The output is a reproducible pair of files:

- `output/profile.json` for machines
- `output/profile.md` for humans

## Learning objectives

By the end, you should be able to:

- validate a CSV input path before reading it
- organize a script with imports, functions, and a dataclass
- compute basic data quality statistics with pandas
- write deterministic JSON and Markdown artifacts
- explain at least 3 data quality findings from the profile


## Output contract

A reliable script should make a clear promise. Given the same input CSV, this notebook's script should always produce the same profile values.

The profile includes:

- row and column counts
- column names and dtypes
- missing values by column
- duplicate row count
- numeric summaries
- top categorical values

It should fail early for missing or empty input files.


## Script mental model

The script is split into small parts:

- imports load libraries
- `Profile` defines the output shape
- `load_csv` validates the file boundary
- `make_profile` computes facts from the DataFrame
- `profile_to_markdown` creates the human-readable report
- `main` connects command-line arguments to the pipeline

This structure makes the script easier to test and easier to debug.


## Exercise 1: Imports and output directory

This cell imports the standard library modules and pandas. It also creates the output directory used by the rest of the notebook.


In [ ]:
import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("output directory:", OUTPUT_DIR.resolve())
print("pandas version:", pd.__version__)


## Exercise 2: Create a controlled sample CSV

Start with known data before testing unknown data. This sample includes missing values and a duplicate row so you can verify that the profiler detects them.


In [ ]:
sample_path = OUTPUT_DIR / "sample_profile.csv"

sample_df = pd.DataFrame(
    {
        "user_id": [1, 2, 3, 4, 4],
        "age": [22, None, 35, 29, 29],
        "country": ["US", "SG", None, "US", "US"],
        "plan": ["free", "pro", "free", "pro", "pro"],
    }
)
sample_df.to_csv(sample_path, index=False)

print("wrote sample:", sample_path)
display(sample_df)


## Exercise 3: Define the profile shape and input validation

A dataclass is a typed container for the output contract. `load_csv` treats the file path as untrusted input and fails before pandas tries to read bad data.


In [ ]:
@dataclass
class Profile:
    rows: int
    cols: int
    columns: List[str]
    dtypes: Dict[str, str]
    missing_by_column: Dict[str, int]
    duplicate_rows: int
    numeric_summary: Dict[str, Dict[str, Optional[float]]]
    categorical_top_values: Dict[str, Dict[str, int]]


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")
    if path.stat().st_size == 0:
        raise ValueError(f"Input file is empty: {path}")
    return pd.read_csv(path)


loaded_df = load_csv(sample_path)
print("loaded shape:", loaded_df.shape)


## Exercise 4: Compute profile statistics

Pandas sometimes returns NumPy numeric types, which are not always JSON-safe. `clean_number` converts numeric summary values into plain Python floats or `None`.


In [ ]:
def clean_number(value) -> Optional[float]:
    if pd.isna(value):
        return None
    return float(value)


def make_profile(df: pd.DataFrame) -> Profile:
    missing = df.isna().sum().to_dict()
    dtypes = {col: str(dtype) for col, dtype in df.dtypes.to_dict().items()}
    numeric_summary = (
        df.select_dtypes(include="number")
        .agg(["min", "max", "mean"])
        .round(4)
        .to_dict()
    )
    categorical_top_values = {
        col: {
            str(value): int(count)
            for value, count in df[col].fillna("<MISSING>").value_counts().head(5).items()
        }
        for col in df.select_dtypes(exclude="number").columns
    }

    return Profile(
        rows=int(df.shape[0]),
        cols=int(df.shape[1]),
        columns=list(df.columns),
        dtypes=dtypes,
        missing_by_column={col: int(count) for col, count in missing.items()},
        duplicate_rows=int(df.duplicated().sum()),
        numeric_summary={
            col: {stat: clean_number(value) for stat, value in stats.items()}
            for col, stats in numeric_summary.items()
        },
        categorical_top_values=categorical_top_values,
    )


profile = make_profile(loaded_df)
profile


## Exercise 5: Write deterministic JSON

`sort_keys=True` keeps JSON key order stable. Stable output makes diffs meaningful when the same input is profiled more than once.


In [ ]:
profile_json_path = OUTPUT_DIR / "profile.json"
json_text = json.dumps(asdict(profile), indent=2, sort_keys=True)
profile_json_path.write_text(json_text, encoding="utf-8")

print("wrote:", profile_json_path)
print(json_text[:500])


## Exercise 6: Write a Markdown report

JSON is useful for code. Markdown is useful for people. Both files describe the same profile, but they serve different readers.


In [ ]:
def profile_to_markdown(profile: Profile) -> str:
    lines = []
    lines.append("# Data Profile")
    lines.append("")
    lines.append(f"- Rows: {profile.rows}")
    lines.append(f"- Columns: {profile.cols}")
    lines.append(f"- Duplicate rows: {profile.duplicate_rows}")
    lines.append("")
    lines.append("## Columns")
    lines.append("")
    lines.append("| column | dtype | missing |")
    lines.append("|---|---|---:|")
    for col in profile.columns:
        dtype = profile.dtypes.get(col, "")
        missing = profile.missing_by_column.get(col, 0)
        lines.append(f"| {col} | {dtype} | {missing} |")
    lines.append("")

    if profile.numeric_summary:
        lines.append("## Numeric Summary")
        lines.append("")
        lines.append("| column | min | max | mean |")
        lines.append("|---|---:|---:|---:|")
        for col, stats in profile.numeric_summary.items():
            lines.append(
                f"| {col} | {stats.get('min', '')} | {stats.get('max', '')} | {stats.get('mean', '')} |"
            )
        lines.append("")

    if profile.categorical_top_values:
        lines.append("## Top Categorical Values")
        lines.append("")
        for col, values in profile.categorical_top_values.items():
            lines.append(f"### {col}")
            for value, count in values.items():
                lines.append(f"- {value}: {count}")
            lines.append("")

    return "\n".join(lines)


profile_md_path = OUTPUT_DIR / "profile.md"
markdown_text = profile_to_markdown(profile)
profile_md_path.write_text(markdown_text, encoding="utf-8")

print("wrote:", profile_md_path)
print(markdown_text)


## Exercise 7: Optional required columns

Some projects require specific columns. Validate those requirements at the boundary so the error message is clear.


In [ ]:
def require_columns(df: pd.DataFrame, required: List[str]) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


require_columns(loaded_df, ["user_id", "age", "country"])
print("required columns present")


## Exercise 8: Assemble a script entrypoint

In a `.py` file, `main()` would parse command-line arguments and run the full pipeline. The notebook uses the same structure so you can move the code into `data_profile.py`.


In [ ]:
def run_profile(input_path: Path, output_dir: Path, required_columns: Optional[List[str]] = None) -> Profile:
    output_dir.mkdir(parents=True, exist_ok=True)
    df = load_csv(input_path)
    if required_columns:
        require_columns(df, required_columns)

    profile = make_profile(df)
    (output_dir / "profile.json").write_text(
        json.dumps(asdict(profile), indent=2, sort_keys=True),
        encoding="utf-8",
    )
    (output_dir / "profile.md").write_text(profile_to_markdown(profile), encoding="utf-8")
    return profile


rerun_profile = run_profile(sample_path, OUTPUT_DIR, required_columns=["user_id", "age"])
print("rows:", rerun_profile.rows)
print("duplicate rows:", rerun_profile.duplicate_rows)


## Required data quality note

After generating the profile, write at least 3 findings in `report.md`. Use evidence from `profile.json` or `profile.md`.

Example findings from the sample data:

- `age` has 1 missing value.
- `country` has 1 missing value.
- There is 1 duplicate row.
- `country` has `US` as the most frequent value.


In [ ]:
report_path = OUTPUT_DIR / "report.md"
report_text = """# Data Quality Findings

- `age` has 1 missing value in `missing_by_column`.
- `country` has 1 missing value in `missing_by_column`.
- The profile reports 1 duplicate row.
"""
report_path.write_text(report_text, encoding="utf-8")
print("wrote:", report_path)
print(report_text)


## Summary

You now have the core pattern for data profiling:

1. Validate inputs at the boundary.
2. Compute profile facts with pandas.
3. Convert values into JSON-safe types.
4. Write stable machine-readable and human-readable outputs.
5. Explain findings with evidence.

## References

- Pandas getting started: https://pandas.pydata.org/docs/getting_started/index.html
- Pandas I/O: https://pandas.pydata.org/docs/user_guide/io.html
- Python `json`: https://docs.python.org/3/library/json.html
- Python `argparse`: https://docs.python.org/3/library/argparse.html
